# Sketch2App: layout LLM fine-tuning (LoRA SFT)

Fine-tunes **Gemma 4 E2B** (primary) or **Qwen3.5-0.8B** (fallback) to turn a sketch's element list into the
canonical UI spec JSON, evaluates it against the rule-based baseline and the base model (few-shot), and exports
the adapter, a merged model, and (optionally) GGUF for LAN mode and `.litertlm` for on-device.

**Runs on:** free Colab T4 (16 GB, QLoRA 4-bit) or Kaggle T4/P100; faster on a rented 24 GB GPU (set `LOAD_4BIT=False`).

**Before you start:** upload `sketch2app_llm_bundle.zip` (made on the laptop by `python -m s2a.layout.sft_data`,
in `D:/sketch2app-data/llm/`) to `/content/` (Colab Files panel) or attach it as a Kaggle dataset.

**Outputs** (in `OUT_DIR`, zip downloaded at the end): `adapter/`, `merged/`, `predictions_*.jsonl`
(score them on the laptop with `python -m s2a.layout.evaluate --method predictions --pred ...`),
`metrics.json`, `versions.json`.

In [ ]:
# ---- configuration -------------------------------------------------------------------------------
MODEL = "unsloth/gemma-4-E2B-it"      # fallback: "unsloth/Qwen3.5-0.8B"
LOAD_4BIT = True                      # QLoRA when the GPU has < 24 GB; False on a 24 GB GPU
MAX_LEN = 2048
TRAIN_EXAMPLES = 8000                 # of ~33k; raise on a faster GPU (time scales linearly)
EPOCHS = 2
LR = 2e-4
LORA_R, LORA_ALPHA = 16, 32
BATCH, GRAD_ACCUM = 4, 4
EVAL_EVERY = 200                      # steps between validation generations (early stopping signal)
EVAL_SAMPLES = 64                     # val prompts generated per check
PATIENCE = 3                          # stop after this many checks without tree-similarity improvement
TEST_SAMPLES = 300                    # test prompts per condition for the final evaluation
BUNDLE = "/content/sketch2app_llm_bundle.zip"
OUT_DIR = "/content/sketch2app_llm_out"
SEED = 1234

In [ ]:
!pip install -q "unsloth==2026.10.3" "jsonschema==4.26.0" "apted==1.0.3"

In [ ]:
import glob, json, os, sys, time, zipfile, importlib.metadata as md
os.makedirs(OUT_DIR, exist_ok=True)
if not os.path.exists(BUNDLE):
    found = glob.glob("/kaggle/input/**/sketch2app_llm_bundle.zip", recursive=True)
    if found:
        BUNDLE = found[0]
    else:
        from google.colab import files
        files.upload()  # pick sketch2app_llm_bundle.zip
zipfile.ZipFile(BUNDLE).extractall("/content/s2a_bundle")
sys.path.insert(0, "/content/s2a_bundle/ml")
from s2a.layout.metrics import score, mean_scores  # same metric code as docs/results.md

def read_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

D = "/content/s2a_bundle/data"
train = read_jsonl(f"{D}/sft_train.jsonl")
val = read_jsonl(f"{D}/sft_val.jsonl")
tests = {c: read_jsonl(f"{D}/eval_test_{c}.jsonl")[:TEST_SAMPLES] for c in ("gold", "recognizer") if os.path.exists(f"{D}/eval_test_{c}.jsonl")}
import random; random.Random(SEED).shuffle(train); train = train[:TRAIN_EXAMPLES]
print(len(train), "train,", len(val), "val,", {k: len(v) for k, v in tests.items()}, "test")
versions = {p: md.version(p) for p in ("unsloth", "transformers", "trl", "peft", "torch", "bitsandbytes", "datasets")}
json.dump(versions, open(f"{OUT_DIR}/versions.json", "w"), indent=1); print(versions)

In [ ]:
import torch
from unsloth import FastModel
model, tokenizer = FastModel.from_pretrained(model_name=MODEL, max_seq_length=MAX_LEN, load_in_4bit=LOAD_4BIT,
                                             full_finetuning=False, dtype=None)
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False, finetune_language_layers=True,   # text only
    finetune_attention_modules=True, finetune_mlp_modules=True,     # all linear layers
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0, bias="none", random_state=SEED,
)
IS_GEMMA = "gemma" in MODEL.lower()
if IS_GEMMA:
    from unsloth.chat_templates import get_chat_template
    tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")   # non-thinking template for small Gemma 4
TOK = getattr(tokenizer, "tokenizer", tokenizer)  # text tokenizer under a multimodal processor
TOK.padding_side = "left"                           # batched generation

In [ ]:
def chat(prompt, answer=None):
    msgs = [{"role": "user", "content": prompt}]
    if answer is not None:
        msgs.append({"role": "assistant", "content": answer})
    kw = {} if IS_GEMMA else {"enable_thinking": False}
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=answer is None, **kw)
    return text.removeprefix("<bos>")  # the trainer adds <bos> itself

from datasets import Dataset
train_ds = Dataset.from_list([{"text": chat(r["prompt"], r["completion"])} for r in train])
print(train_ds[0]["text"][:600])

In [ ]:
@torch.no_grad()
def generate(prompts, batch=16, max_new_tokens=1024):
    # Greedy decoding in batches; returns (texts, ms per prompt).
    FastModel.for_inference(model)
    outs, t0 = [], time.time()
    for i in range(0, len(prompts), batch):
        texts = [chat(p) for p in prompts[i:i + batch]]
        enc = TOK(texts, return_tensors="pt", padding=True, add_special_tokens=True).to("cuda")
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
        outs += TOK.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    FastModel.for_training(model)
    return outs, (time.time() - t0) * 1000 / max(1, len(prompts))

def evaluate(rows, outputs):
    return mean_scores([score(o, r.get("gold") or json.loads(r["completion"])) for r, o in zip(rows, outputs)])

In [ ]:
from transformers import TrainerCallback
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

class TreeSimilarityEarlyStop(TrainerCallback):
    # Every EVAL_EVERY steps: generate EVAL_SAMPLES val specs, score tree similarity, keep the best adapter,
    # stop after PATIENCE checks without improvement (the brief's early-stopping criterion).
    def __init__(self):
        self.best, self.bad, self.history = -1.0, 0, []
    def on_step_end(self, args, state, control, **kw):
        if state.global_step % EVAL_EVERY:
            return
        outs, _ = generate([r["prompt"] for r in val[:EVAL_SAMPLES]])
        m = evaluate(val[:EVAL_SAMPLES], outs)
        self.history.append({"step": state.global_step, **m}); print(self.history[-1])
        if m["tree_similarity"] > self.best:
            self.best, self.bad = m["tree_similarity"], 0
            model.save_pretrained(f"{OUT_DIR}/adapter"); tokenizer.save_pretrained(f"{OUT_DIR}/adapter")
        else:
            self.bad += 1
            control.should_training_stop = self.bad >= PATIENCE

early = TreeSimilarityEarlyStop()
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_ds, callbacks=[early],
    args=SFTConfig(dataset_text_field="text", max_length=MAX_LEN, per_device_train_batch_size=BATCH,
                   gradient_accumulation_steps=GRAD_ACCUM, num_train_epochs=EPOCHS, learning_rate=LR,
                   warmup_steps=20, lr_scheduler_type="cosine", optim="adamw_8bit", weight_decay=0.01,
                   logging_steps=20, save_strategy="no", seed=SEED, report_to="none", output_dir=f"{OUT_DIR}/trainer"),
)
if IS_GEMMA:
    trainer = train_on_responses_only(trainer, instruction_part="<|turn>user\n", response_part="<|turn>model\n")
else:
    trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
stats = trainer.train()
json.dump(early.history, open(f"{OUT_DIR}/val_history.json", "w"), indent=1)

In [ ]:
# Reload the best adapter (by validation tree similarity) and evaluate on the test prompts.
model.load_adapter(f"{OUT_DIR}/adapter", adapter_name="best"); model.set_adapter("best")
results = {"model": MODEL, "train_examples": len(train), "best_val_tree_similarity": early.best}
for cond, rows in tests.items():
    outs, ms = generate([r["prompt"] for r in rows])
    with open(f"{OUT_DIR}/predictions_finetuned_{cond}.jsonl", "w", encoding="utf-8") as f:
        for r, o in zip(rows, outs):
            f.write(json.dumps({"id": r["id"], "output": o, "ms": ms}) + "\n")
    results[f"finetuned_{cond}"] = {**evaluate(rows, outs), "ms_per_sample": ms}
    print(cond, results[f"finetuned_{cond}"])

In [ ]:
# Baseline: the same base model without the adapter, prompted with 3 solved examples (few-shot).
shots = sorted(train, key=lambda r: len(r["prompt"]) + len(r["completion"]))[:3]
prefix = "".join(f"Example input:\n{s['prompt']}\nExample output:\n{s['completion']}\n\n" for s in shots)
with model.disable_adapter():
    for cond, rows in tests.items():
        outs, ms = generate([prefix + "Now this input:\n" + r["prompt"] for r in rows], batch=8)
        with open(f"{OUT_DIR}/predictions_fewshot_{cond}.jsonl", "w", encoding="utf-8") as f:
            for r, o in zip(rows, outs):
                f.write(json.dumps({"id": r["id"], "output": o, "ms": ms}) + "\n")
        results[f"fewshot_{cond}"] = {**evaluate(rows, outs), "ms_per_sample": ms}
        print(cond, results[f"fewshot_{cond}"])
json.dump(results, open(f"{OUT_DIR}/metrics.json", "w"), indent=1)

In [ ]:
# Merged 16-bit model: input for GGUF (LAN mode) and LiteRT-LM (on-device) conversion.
model.save_pretrained_merged(f"{OUT_DIR}/merged", tokenizer, save_method="merged_16bit")

In [ ]:
# Optional, LAN mode: GGUF for llama.cpp on the laptop (4 GB GPU / CPU).
model.save_pretrained_gguf(f"{OUT_DIR}/gguf", tokenizer, quantization_method="q4_k_m")

In [ ]:
# Optional, on-device: LiteRT-LM bundle (Google's documented path; litert-torch needs Linux, which Colab is).
!pip install -q "litert-torch-nightly==0.10.0.dev20261009" "litert-lm==0.18.0"
!litert-torch export_hf --model={OUT_DIR}/merged --output_dir={OUT_DIR}/litertlm --externalize_embedder
!ls -la {OUT_DIR}/litertlm

In [ ]:
# Small results as a zip download; large artefacts (merged / gguf / litertlm) go to Google Drive if mounted.
import shutil
small = "/content/sketch2app_llm_results"
os.makedirs(small, exist_ok=True)
for name in os.listdir(OUT_DIR):
    src = os.path.join(OUT_DIR, name)
    if name in ("adapter",) or name.endswith((".json", ".jsonl")):
        (shutil.copytree if os.path.isdir(src) else shutil.copy)(src, os.path.join(small, name))
shutil.make_archive(small, "zip", small)
try:
    from google.colab import drive, files
    drive.mount("/content/drive")
    for big in ("merged", "gguf", "litertlm"):
        if os.path.isdir(f"{OUT_DIR}/{big}"):
            shutil.copytree(f"{OUT_DIR}/{big}", f"/content/drive/MyDrive/sketch2app/{big}", dirs_exist_ok=True)
    files.download(f"{small}.zip")
except ImportError:
    print("Kaggle: download the zip and the large folders from the output panel")